# 2.2 Vector Search with ChromaDB

**Store embeddings and search by meaning, not keywords.**

In this notebook you will:
- Understand what a vector database is and why it's needed
- Create a ChromaDB collection and add documents
- Perform semantic search (find documents by meaning)
- Compare semantic search vs keyword search
- Use metadata filtering to narrow results

> **No API key needed!** Embeddings run locally with sentence-transformers.

## 1. Setup

We need:
- **chromadb**: an open-source vector database that runs locally
- **sentence-transformers**: to create embeddings (same model from notebook 2.1)

In [ ]:
# Install required packages quietly
!pip install chromadb sentence-transformers -q

## 2. What is a Vector Database?

A **regular database** stores data in rows and columns. You search with exact queries:
```sql
SELECT * FROM flights WHERE destination = 'Tokyo'
```

A **vector database** stores data as **embedding vectors**. You search with **meaning**:
```
"flights going to Japan" → finds documents about Tokyo, Osaka, Narita...
```

It works in 3 steps:
1. **Store**: Convert documents to embeddings and save them
2. **Query**: Convert your question to an embedding
3. **Search**: Find stored embeddings closest to your question

**ChromaDB** is a popular open-source vector database that's perfect for learning — it runs entirely in memory, no server needed.

## 3. Create a ChromaDB Collection

A **collection** in ChromaDB is like a table in a regular database. It holds documents, their embeddings, and optional metadata.

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# Create an in-memory ChromaDB client.
# In-memory means data is lost when the notebook restarts.
# For persistence, you'd use: chromadb.PersistentClient(path="./chroma_db")
client = chromadb.Client()

# Tell ChromaDB to use the same embedding model we used in notebook 2.1.
# ChromaDB will automatically embed documents when we add them.
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create a collection — like creating a table in a database.
# If it already exists, get_or_create avoids an error.
collection = client.get_or_create_collection(
    name="travel_docs",
    embedding_function=embedding_fn
)

print(f"Collection '{collection.name}' created successfully!")
print(f"Current document count: {collection.count()}")

## 4. Add Documents

Let's add travel-related documents. Each document gets:
- A unique **id** (required by ChromaDB)
- The **document** text (ChromaDB will automatically embed it)
- **Metadata** (optional key-value pairs for filtering)

In [ ]:
# Define our travel knowledge base
documents = [
    "Free cancellation is available up to 24 hours before departure for all economy tickets.",
    "Business class passengers get priority boarding and access to the airport lounge.",
    "Baggage allowance for international flights is 2 checked bags of 23kg each.",
    "Hotel check-in time is 3:00 PM and check-out is at 11:00 AM.",
    "Airport transfer service is available from all major hotels for $25 per person.",
    "Travel insurance covers medical emergencies, trip cancellation, and lost baggage.",
    "Visa-free entry is available for Turkish citizens traveling to Japan for up to 90 days.",
    "The loyalty program offers 2x miles on all international flights booked directly.",
    "Child discount of 25% applies to passengers aged 2-11 on all routes.",
    "Flight rebooking is free for premium members, others pay a $50 change fee.",
]

# Metadata helps us filter results later.
# Each document is tagged with a category.
metadatas = [
    {"category": "booking", "topic": "cancellation"},
    {"category": "service", "topic": "business_class"},
    {"category": "baggage", "topic": "allowance"},
    {"category": "hotel", "topic": "check_in"},
    {"category": "service", "topic": "transfer"},
    {"category": "booking", "topic": "insurance"},
    {"category": "travel", "topic": "visa"},
    {"category": "booking", "topic": "loyalty"},
    {"category": "booking", "topic": "discount"},
    {"category": "booking", "topic": "rebooking"},
]

# Unique IDs for each document (required by ChromaDB)
ids = [f"doc_{i}" for i in range(len(documents))]

# Add all documents to the collection.
# ChromaDB automatically embeds each document using our embedding function.
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Added {collection.count()} documents to the collection.")

## 5. Semantic Search

Now the powerful part: we can search by **meaning**, not exact words.

ChromaDB will:
1. Embed our query using the same model
2. Find the documents whose embeddings are closest to the query
3. Return them ranked by similarity

In [ ]:
# Search for documents related to our query.
# n_results controls how many documents to return.
results = collection.query(
    query_texts=["Can I cancel my ticket?"],  # Our natural language question
    n_results=3  # Return top 3 most similar documents
)

# Display the results nicely
print("Query: 'Can I cancel my ticket?'")
print("=" * 60)
for i, (doc, distance) in enumerate(zip(results["documents"][0], results["distances"][0])):
    # ChromaDB returns distances — lower = more similar.
    # We convert to a similarity score (1 - distance/2) for easier reading.
    similarity = 1 - distance / 2
    print(f"\nResult {i+1} (similarity: {similarity:.3f}):")
    print(f"  {doc}")

In [ ]:
# Let's try more queries to see semantic search in action
test_queries = [
    "How much luggage can I bring?",          # Should find baggage policy
    "Do I need a visa for Tokyo?",             # Should find visa info
    "What perks do premium flyers get?",        # Should find business class + loyalty
    "How do I get from the hotel to the airport?",  # Should find transfer service
]

for query in test_queries:
    # Query the collection for each test question
    results = collection.query(query_texts=[query], n_results=2)
    print(f"\nQuery: '{query}'")
    print("-" * 50)
    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        similarity = 1 - dist / 2
        print(f"  [{similarity:.3f}] {doc}")

## 6. Keyword Search vs Semantic Search

Let's see why semantic search is better than simple keyword matching. We'll compare both approaches on the same queries.

In [ ]:
def keyword_search(query, documents, top_k=3):
    """
    Simple keyword search: count how many query words appear in each document.
    This is how traditional search works — exact word matching.
    """
    # Split query into individual words, lowercase for fair comparison
    query_words = set(query.lower().split())
    scores = []
    for doc in documents:
        # Count how many query words appear in this document
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words)  # Set intersection
        scores.append(overlap)
    
    # Sort by score (highest first) and return top_k results
    ranked = sorted(zip(scores, documents), reverse=True)
    return ranked[:top_k]


# A query where semantic search shines:
# "luggage" doesn't appear in any document, but "baggage" does!
query = "How much luggage can I bring on my trip?"

print(f"Query: '{query}'")
print("\n" + "=" * 60)
print("KEYWORD SEARCH RESULTS:")
print("=" * 60)
keyword_results = keyword_search(query, documents, top_k=3)
for score, doc in keyword_results:
    print(f"  [word overlap: {score}] {doc}")

print("\n" + "=" * 60)
print("SEMANTIC SEARCH RESULTS:")
print("=" * 60)
semantic_results = collection.query(query_texts=[query], n_results=3)
for doc, dist in zip(semantic_results["documents"][0], semantic_results["distances"][0]):
    similarity = 1 - dist / 2
    print(f"  [similarity: {similarity:.3f}] {doc}")

print("\n---")
print("Notice: Keyword search can't find 'baggage' when you say 'luggage'.")
print("Semantic search understands they mean the same thing!")

## 7. Metadata Filtering

Sometimes you want to search **within a specific category**. ChromaDB lets you filter by metadata before searching.

This is like adding a `WHERE` clause to your semantic search.

In [ ]:
# Search ONLY within booking-related documents
# The where clause filters by metadata BEFORE semantic search happens
booking_results = collection.query(
    query_texts=["How can I save money?"],
    n_results=3,
    where={"category": "booking"}  # Only search booking documents
)

print("Query: 'How can I save money?' (filtered to category=booking)")
print("=" * 60)
for doc, meta in zip(booking_results["documents"][0], booking_results["metadatas"][0]):
    print(f"  [{meta['topic']}] {doc}")

print("\n" + "=" * 60)

# Now search WITHOUT the filter — see the difference
all_results = collection.query(
    query_texts=["How can I save money?"],
    n_results=3
)

print("\nQuery: 'How can I save money?' (NO filter)")
print("=" * 60)
for doc, meta in zip(all_results["documents"][0], all_results["metadatas"][0]):
    print(f"  [{meta['category']}/{meta['topic']}] {doc}")

print("\nMetadata filtering narrows your search to relevant categories.")

In [ ]:
# You can also use more complex filters with $and, $or operators
# Find service-related documents about either transfer or business class
complex_results = collection.query(
    query_texts=["premium travel experience"],
    n_results=5,
    where={
        "$or": [
            {"category": "service"},
            {"topic": "loyalty"}
        ]
    }
)

print("Query: 'premium travel experience' (category=service OR topic=loyalty)")
print("=" * 60)
for doc, meta in zip(complex_results["documents"][0], complex_results["metadatas"][0]):
    print(f"  [{meta['category']}/{meta['topic']}] {doc}")

## 8. YOUR TURN: Build Your Own Knowledge Base

Create a collection on a topic you care about. Ideas:
- Company FAQ
- Product catalog
- Movie/book descriptions
- Technical documentation

In [ ]:
# YOUR TURN: Create your own collection and search it!

# Step 1: Create a new collection
my_collection = client.get_or_create_collection(
    name="my_knowledge_base",
    embedding_function=embedding_fn
)

# Step 2: Add your own documents (replace these!)
my_docs = [
    "Replace with your first document",
    "Replace with your second document",
    "Replace with your third document",
]

my_collection.add(
    documents=my_docs,
    ids=[f"my_doc_{i}" for i in range(len(my_docs))]
)

# Step 3: Search your collection
my_query = "Replace with your search query"
my_results = my_collection.query(query_texts=[my_query], n_results=2)

print(f"Query: '{my_query}'")
for doc in my_results["documents"][0]:
    print(f"  → {doc}")

## Key Takeaways

1. **Vector databases** store text as embeddings and search by meaning
2. **ChromaDB** is easy to use — create a collection, add documents, query
3. **Semantic search** finds relevant results even when words don't match ("luggage" → "baggage")
4. **Metadata filtering** lets you narrow search to specific categories

**Next up:** In notebook **3.1**, we'll combine vector search with an LLM to build a **complete RAG pipeline** that can answer questions using your documents!